In [ ]:
# =============================================================================
# 1. Importación de Librerías & Cliente
# =============================================================================
import pandas as pd
from datetime import datetime, timedelta
import pytz
from google.cloud import bigquery
from google.cloud import storage
from google.api_core.exceptions import NotFound
clientBQ = bigquery.Client()
storage_client = storage.Client()

In [ ]:
# =============================================================================
# 2. Configuración de Fechas D-1 & Rutas
# =============================================================================

Zona = pytz.timezone('America/Lima')
peru_time = datetime.now(Zona)
peru_time_ayer = peru_time - timedelta(days=1)
var_fecha_ini = peru_time_ayer.strftime('%Y-%m-%d')
var_fecha_fin = peru_time.strftime('%Y-%m-%d')
fecha_fin_dt = datetime.strptime(var_fecha_fin, '%Y-%m-%d')

print(f"--- Fecha de inicio: {var_fecha_ini} ---")
print(f"--- Fecha de Fin: {var_fecha_fin} ---")
print(f"--- Fecha del proceso: {fecha_fin_dt} ---")


--- Fecha de inicio: 2026-04-27 ---
--- Fecha de Fin: 2026-04-28 ---
--- Fecha del proceso: 2026-04-28 00:00:00 ---


In [ ]:
## Variables de fecha como DataEntry si se quiere reprocesar
##var_fecha_ini = '2026-05-01'
##var_fecha_fin = '2026-05-02'
##fecha_fin_dt = datetime.strptime(var_fecha_fin, '%Y-%m-%d')

##print(f"--- Fecha de inicio 1: {var_fecha_ini} ---")
##print(f"--- Fecha de Fin 2: {var_fecha_fin} ---")
##print(f"--- Fecha del proceso 3: {fecha_fin_dt} ---")

--- Fecha de inicio 1: 2026-03-16 ---
--- Fecha de Fin 2: 2026-04-01 ---
--- Fecha del proceso 3: 2026-04-01 00:00:00 ---


In [ ]:
# =============================================================================
# 2. Configuración de Variables de entorno
# =============================================================================
var_anho = fecha_fin_dt.strftime('%Y')
var_mes = fecha_fin_dt.strftime('%m')
var_fecha_file = fecha_fin_dt.strftime('%Y%m%d')

print(f"--- Año del proceso {var_anho} ---")
print(f"--- Mes del proceso: {var_mes} ---")
print(f"--- Fecha del archivo: {var_fecha_file} ---")

--- Año del proceso 2026 ---
--- Mes del proceso: 04 ---
--- Fecha del archivo: 20260401 ---


In [ ]:
# =============================================================================
# 3. Configuración de Rutas y Parámetros
# =============================================================================
bucket_name = "adls-reportes"
ruta_base = f"Data/APA/t_abono_detalle/{var_anho}/{var_mes}/"
nombre_final = f"t_abono_detalle_{var_fecha_file}.parquet"
proyecto = "prd-izipay-data-storage-pv"

In [ ]:
# =============================================================================
# 4. Creación de Tabla Temporal en BigQuery
# =============================================================================
# Se materializa primero el resultado (joins + decryption) en BQ
# para no tener que hacerlo en memoria al momento de exportar.

temp_table_id = f"prd-izipay-data-operation.master_stage_financial.temp_abono_detalle_{var_fecha_file}"

query_temp = f"""
CREATE OR REPLACE TABLE `{temp_table_id}` AS
with aux_iden_party_data_control as (
  select
    party_id_izi,
    document_number
  from prd-izipay-data-sensitive.master_pii.iden_party_data_control
  qualify row_number() over (partition by party_id_izi order by document_number desc ) = 1
)
SELECT
  a.process_date,
  a.itc_company_id,
  a.itc_company_name,
  flujo,
  producto,
  cod_comercio,
  cod_transaccion,
  fecha_proceso,
  cod_banco,
  tipo_pago,
  trim(AEAD.DECRYPT_STRING(b.key, a.cuenta_abono, b.constant)) as cuenta_abono,
  hash_cuenta_abono,
  trim(AEAD.DECRYPT_STRING(b.key, a.cuenta_corriente, b.constant)) as cuenta_corriente,
  hash_cta_corriente,
  cod_moneda,
  importe,
  comision_abono,
  igv_comision,
  neto_1,
  cobro_devolucion,
  neto_2,
  importe_retenido,
  neto_2_dolar,
  neto_3,
  tipo_cambio,
  fecha_abono,
  tipo_cuenta,
  pago_tercero,
  ruc_tercero,
  trim(AEAD.DECRYPT_STRING(c.key, a.nombre_tercero, c.constant)) as nombre_tercero,
  tipo_doc,
  tipo_ruc,
  d.document_number as nro_documento,
  situacion,
  usuario_actualiza,
  mensaje,
  fecha_abono_mod,
  nro_dias_abono,
  sist_comp_tercero,
  cantidad,
  fac_estab,
  fecha_abono_referencial,
  nombre_cheque,
  agrupacion_abonos,
  fuerza_cuenta,
  nro_archivo_abono,
  cci,
  tipo_doc_tercero,
  cuenta_especial,
  cod_padre,
  tipo_facilitador,
  cod_estab_abono,
  nombre_comercial,
  cod_facilitador,
  tipo_doc_identificador,
  tipo_pendiente,
  fecha_entrante_DCP,
  ajuste_DCP,
  motivo_DCP,
  observacion,
  impuesto_emisor,
  cod_dcp,
  ind_capt_proces,
  filtro_abono,
  importe_solarizado,
  dq_flag_ind,
  dq_control_msg,
  dq_config_id,
  cast(format_datetime('%Y-%m-%d %H:%M:%S', cast(a.start_date as datetime)) as string) as start_date,
  cast(format_datetime('%Y-%m-%d %H:%M:%S', cast(a.end_date as datetime)) as string) as end_date,
  flag_active,
  a.record_source,
  cast(format_datetime('%Y-%m-%d %H:%M:%S', cast(a.load_date as datetime)) as string) as load_date,
  a.creation_user,
  producto_abono_det,
  tipo_abono,
  detalle_abono,
  flujo_fuente,
  flag_abono_comercio,
  des_producto,
  nombre_banco,
  des_moneda,
  des_situacion
FROM `prd-izipay-data-storage-pv.master_financial.t_abono_detalle` a
inner join prd-izipay-data-sensitive.secure_secrets.config_protected_data b on (1=1 and b.code = 'C_ACCOUNT_NUMBER')
inner join prd-izipay-data-sensitive.secure_secrets.config_protected_data c on (1=1 and c.code = 'C_BUSINESS_NAME')
left join aux_iden_party_data_control d on (d.party_id_izi = a.party_id_izi)
WHERE a.process_date >=  DATE '{var_fecha_ini}'
  AND a.process_date <=  DATE '{var_fecha_fin}'
"""

print(f"Creando tabla temporal: {temp_table_id} ...")
clientBQ.query(query_temp).result()
print("✅ Tabla temporal creada correctamente.")


Creando tabla temporal: prd-izipay-data-operation.master_stage_financial.temp_abono_detalle_20260401 ...


Forbidden: 403 GET https://bigquery.googleapis.com/bigquery/v2/projects/prd-izipay-data-operation/queries/d36e8519-fd8e-45f6-973a-6a0099eabac4?maxResults=0&location=US&prettyPrint=false: Access Denied: BigQuery BigQuery: User has neither fine-grained reader nor masked get permission to get data protected by policy tag "Izi_Politica_Mask : Alta" on columns prd-izipay-data-sensitive.secure_secrets.config_protected_data.constant, prd-izipay-data-sensitive.secure_secrets.config_protected_data.key.

Location: US
Job ID: d36e8519-fd8e-45f6-973a-6a0099eabac4


In [ ]:
# =============================================================================
# 5. Exportación desde Tabla Temporal a GCS (Parquet)
# =============================================================================
# Se exporta directo desde BQ a GCS sin pasar por memoria Python.

uri_temporal = f"gs://{bucket_name}/{ruta_base}temp_{var_fecha_file}_*.parquet"
print(f"Exportando desde tabla temporal a: {uri_temporal} ...")

query_export = f"""
EXPORT DATA OPTIONS (
  uri = '{uri_temporal}',
  format = 'PARQUET',
  overwrite = true
) AS
SELECT * FROM `{temp_table_id}`;
"""

clientBQ.query(query_export).result()
print("✅ Exportación a GCS completada.")

# Eliminar tabla temporal
##clientBQ.delete_table(temp_table_id)
##print(f"🧹 Tabla temporal eliminada: {temp_table_id}")


In [ ]:
# =============================================================================
# 6. Consolidación incremental con PyArrow (sin cargar todo en memoria)
# =============================================================================
import pyarrow.parquet as pq
import pyarrow as pa

print("Consolidando archivos en uno solo (modo incremental)...")
bucket = storage_client.bucket(bucket_name)
prefix_temp = f"{ruta_base}temp_{var_fecha_file}_"

blobs = list(bucket.list_blobs(prefix=prefix_temp))

if not blobs:
    print("⚠️ AVISO: No se encontraron archivos temporales para consolidar.")
else:
    ruta_final_full = f"gs://{bucket_name}/{ruta_base}{nombre_final}"
    writer = None

    for blob in blobs:
        uri_parte = f"gs://{bucket_name}/{blob.name}"
        table = pq.read_table(uri_parte)  # lee un part a la vez
        if writer is None:
            writer = pq.ParquetWriter(ruta_final_full, table.schema)
        writer.write_table(table)
        del table  # libera memoria inmediatamente

    if writer:
        writer.close()

    # Borrar temporales
    for blob in blobs:
        try:
            bucket.blob(blob.name).delete()
        except Exception as e:
            print(f"⚠️ No se pudo borrar {blob.name}: {e}")

    print(f"✅ ÉXITO: Archivo único creado en: {ruta_final_full}")
    print(f"🧹 {len(blobs)} archivos temporales eliminados")